# Agentic X — Testing & Evaluation — runnable MOCK checks

**CAI461 | Studio 67 | Parallel team development**

This notebook includes **runnable mock-code cells**. It is not a completed AI or MAS implementation.

## Purpose & limitations

> **MOCK DEVELOPMENT MODE — NOT REAL AI.** These cells can run before Notebook 04 exists. All results are synthetic and must be replaced with actual engine results before evaluation and submission.

**Dependencies:** the shared `core/mock_engine.py` module committed in this GitHub repository. No API key or LangSmith is required for these mock cells.

**Team rule:** Work on your own branch; don't treat MOCK output as actual model performance.

**Notebook 06 responsibility:** build the evaluation harness and LangSmith evaluation plan. This early version checks only **3 synthetic demo fixtures**, NOT the 150-case workbook, and does not call LangSmith.

## Connect to shared engine

Use the shared mock module.

In [ ]:
# Clone this repository into the fresh Colab runtime if needed.
import os, sys, subprocess
repo_dir = "/content/Agentic-X" if os.path.isdir("/content") else os.path.abspath("Agentic-X")
if not os.path.isfile(os.path.join(repo_dir, "core", "mock_engine.py")):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/RENADBHAI/Agentic-X.git", repo_dir],
        check=True,
    )
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
from core.mock_engine import MockEngine, MOCK_CASES, get_mock_case, run_scenario
engine = MockEngine()
print("MOCK engine ready —", len(MOCK_CASES), "demo scenarios only")

## Run smoke checks

These checks assert output schema, message structure, valid decisions and mock labeling. They are **tests of the stub**, not MAS quality measurements.

In [ ]:
required = {"run_id", "case_id", "status", "selected_option_id", "reason",
            "rounds", "agent_evaluations", "messages", "events", "is_mock"}
smoke_results = []
for case_id in MOCK_CASES:
    result = run_scenario(case_id, engine)
    assert required <= set(result)
    assert result["is_mock"] is True
    assert result["status"] in {"AGREED", "ESCALATED", "FAILED"}
    assert result["messages"]
    assert all(m["case_id"] == case_id for m in result["messages"])
    assert all(m["performative"] in {"REQUEST", "INFORM", "CFP", "PROPOSE", "ACCEPT", "REFUSE"}
               for m in result["messages"])
    option_ids = {opt["option_id"] for opt in get_mock_case(case_id)["options"]}
    assert result["selected_option_id"] is None or result["selected_option_id"] in option_ids
    smoke_results.append((case_id, result["status"]))
print("MOCK smoke tests passed:", len(smoke_results))
for record in smoke_results:
    print(*record)

## Check safety veto and forced escalation

Demonstrate two useful acceptance assertions in the mock engine.

In [ ]:
safety_result = run_scenario("SFT-DEMO", engine)
assert safety_result["selected_option_id"] != "SFT-D-O1", "Unsafe option must never be selected"
forced = run_scenario("SFT-DEMO", engine, overrides={"budget_sar": 0, "time_left_min": 0})
assert forced["status"] == "ESCALATED" and forced["selected_option_id"] is None
print("MOCK safety and escalation assertions passed.")

## Prepare real dataset / LangSmith evaluation

After the real engine is available: import 150 workbook cases, run reference constraints separately, configure LangSmith tracing/datasets, evaluate legality, incident resolution, escalation, latency, LLM quality and failure handling. No benchmark is claimed here.